# 🧪 MA-GRPO: Chạy Thử 3 Prompt Từ MoPS → Sinh 24 Sample Data (`Condition D`, Không Freytag)
### Kiểm tra thực tế chất lượng dàn ý, `context_summary`, `prev_chapter_tail` và `prompt` train GRPO trước khi chạy full 80 truyện

- **Input:** 3 đề bài đầu tiên từ `data/mops_premises.jsonl` (*Romance, Fantasy, Contemporary*).
- **Model sinh data (Outline + Transition Prose + Coordinator):** `Qwen/Qwen2.5-32B-Instruct-AWQ` (vLLM `float16`, batch `3` truyện song song).
- **Kiến trúc dàn ý:** **Condition D (Macro Event Graph thuần nhân quả, bỏ Freytag 5 pha, bỏ SubEventAgent)**.
- **Đầu ra:** `24` sample chapter prompts (`3 truyện × 8 chương`) lưu tại `/content/drive/MyDrive/MA-GRPO-Dataset-Sample3/grpo_sample_3mops.jsonl` + file đọc thử `sample_3mops_readable_preview.md`.
- **Thời gian dự kiến trên Colab A100:** ~3–5 phút.

---

## Bước 1: Kiểm tra GPU, Mount Drive & Clone `MA-GRPO`

In [ ]:
!nvidia-smi

import os
import sys
import subprocess
import getpass
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/MA-GRPO-Dataset-Sample3")
except Exception:
    OUTPUT_DIR = Path("sample_3mops_outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục lưu sample data: {OUTPUT_DIR}")

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"

if os.path.exists("/content"):
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    except Exception:
        token = None

    if not os.path.exists(REPO_DIR):
        if token:
            subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
        else:
            res = subprocess.run(["git", "clone", f"https://{REPO_URL}", REPO_DIR])
            if res.returncode != 0:
                token = getpass.getpass("Nhập GitHub Personal Access Token: ")
                subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

assert os.path.exists("scripts/generate_grpo_dataset.py"), "Không tìm thấy scripts/generate_grpo_dataset.py!"
print("Working dir:", os.getcwd())
!git log -n 1 --oneline


## Bước 2: Cài đặt `vllm` & Thư viện

In [ ]:
!pip install -q vllm pandas pyyaml pydantic
print("✅ Cài đặt hoàn tất!")


## Bước 3: Xem Trước 3 Prompt Đầu Vào Lấy Từ `data/mops_premises.jsonl`

In [ ]:
from src.dataset_generator import load_premises_pool

sample_premises, excluded = load_premises_pool(
    premises_files=["data/mops_premises.jsonl"],
    include_canonical=False,
    exclude_files=["data/tell-me-a-story-test.jsonl"],
    num_premises=3,
    target_words_choices=(8000, 9000, 10000),
    num_chapters=8,
)

print(f"Đã chọn {len(sample_premises)} đề bài MoPS (đã loại {excluded} trùng tập test):\n")
for i, p in enumerate(sample_premises, 1):
    print(f"[{i}] ID: {p['id']} | Genre: {p['genre']} | Target: {p['target_words']}w / {p['num_chapters']} chs")
    print(f"    Premise: {p['premise']}\n")


## Bước 4: Chạy Sinh Sample Data (`3` Truyện × `8` Chương = `24` Sample Prompts)
- `VALIDATOR_THRESHOLD = 0.0`: **tắt Heuristic Gate** để giữ trọn bản tóm tắt giàu ý của `Qwen2.5-32B-Instruct-AWQ` *(đổi thành `0.80` nếu muốn so sánh khi bật Heuristic Gate)*.
- Chạy `batch_size = 3` (cả 3 truyện song song trên A100) với log streaming trực tiếp.

In [ ]:
import subprocess
import sys

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
NUM_PREMISES = 3
NUM_CHAPTERS = 8
BATCH_SIZE = 3
VALIDATOR_THRESHOLD = 0.0     # 0.0 = tin JSON 32B-AWQ (tránh Heuristic bắt nhầm paraphrase); đổi 0.80 nếu muốn test Gate
OUTPUT_FILENAME = "grpo_sample_3mops.jsonl"

cmd = [
    sys.executable, "-u", "scripts/generate_grpo_dataset.py",
    "--backend", "vllm",
    "--model", MODEL_ID,
    "--premises-file", "data/mops_premises.jsonl",
    "--no-canonical",
    "--exclude-test-file", "data/tell-me-a-story-test.jsonl",
    "--num-premises", str(NUM_PREMISES),
    "--num-chapters", str(NUM_CHAPTERS),
    "--target-words-choices", "8000", "9000", "10000",
    "--batch-size", str(BATCH_SIZE),
    "--gpu-memory-utilization", "0.88",
    "--max-model-len", "12288",
    "--prose-max-tokens", "900",
    "--validator-threshold", str(VALIDATOR_THRESHOLD),
    "--output-dir", str(OUTPUT_DIR),
    "--output-filename", OUTPUT_FILENAME,
]

print("Running command:\n", " ".join(cmd), "\n", flush=True)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"generate_grpo_dataset.py exited with code {ret}")
print("\n✅ Đã sinh xong 24 sample chapter prompts!")


## Bước 5: Soi Chi Tiết 24 Sample Data Vừa Sinh & Xuất File Markdown Đọc Thử

In [ ]:
import json
import pandas as pd
from pathlib import Path
from scripts.train import (
    extract_context_from_prompt,
    extract_tail_from_prompt,
    extract_target_words_from_text,
)

jsonl_path = OUTPUT_DIR / OUTPUT_FILENAME
report_path = OUTPUT_DIR / "dataset_quality_report.json"

print("=== QUALITY REPORT ===")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))

rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"\nTổng số sample rows: {len(rows)}")

table_rows = []
for r in rows:
    vm = r.get("validation_metadata") or {}
    prompt_txt = r["prompt"]
    table_rows.append({
        "sample_id": r["sample_id"][:16] + "..." if len(r["sample_id"]) > 18 else r["sample_id"],
        "genre": r.get("genre"),
        "ch": r.get("chapter_index"),
        "target_w": r.get("target_words"),
        "parser_ok": extract_target_words_from_text(prompt_txt) == r.get("target_words"),
        "ctx_w": len(str(r.get("context_summary", "")).split()),
        "tail_w": len(str(r.get("prev_chapter_tail", "")).split()),
        "V_s": vm.get("composite_score"),
        "fallback": vm.get("is_fallback"),
        "no_freytag": "Narrative Arc Stage:" not in prompt_txt,
    })

df = pd.DataFrame(table_rows)
print("\n=== BẢNG TỔNG QUAN 24 SAMPLE CHAPTER PROMPTS ===")
print(df.to_string(index=False))

# Xuất file Markdown dễ đọc để kiểm tra trực tiếp bằng mắt trên Drive
md_lines = ["# Sample Data Preview: 3 MoPS Premises (Condition D, No Freytag)\n"]
seen_pids = []
for r in rows:
    pid = r["premise_id"]
    if pid not in seen_pids:
        seen_pids.append(pid)
        md_lines.append(f"\n---\n## Story {len(seen_pids)}: `{pid}` ({r['genre']})\n")
    if r["chapter_index"] in (1, 2, 5, 8):
        md_lines.append(f"### Chapter {r['chapter_index']} — {r['chapter_title']} (target={r['target_words']}w)\n")
        md_lines.append(f"**context_summary:**\n> {r['context_summary']}\n")
        if r["prev_chapter_tail"]:
            md_lines.append(f"**prev_chapter_tail ({len(r['prev_chapter_tail'].split())} words):**\n> {r['prev_chapter_tail']}\n")
        md_lines.append(f"**Full GRPO Training Prompt:**\n```text\n{r['prompt']}\n```\n")

preview_md = OUTPUT_DIR / "sample_3mops_readable_preview.md"
preview_md.write_text("\n".join(md_lines), encoding="utf-8")
print(f"\n✅ Đã xuất bản đọc thử đầy đủ tại: {preview_md}")

# In trực tiếp Mẫu Chương 1 & Chương 2 của truyện đầu tiên ngay trong cell
print("\n" + "=" * 75)
print("PREVIEW TRỰC TIẾP: TRUYỆN 1 — CHƯƠNG 2 (CÓ CONTEXT SUMMARY + PREV_CHAPTER_TAIL THẬT)")
print("=" * 75)
ch2 = next((r for r in rows if r["chapter_index"] == 2), rows[0])
print(ch2["prompt"])


## Bước 6: Tự động Ngắt Server Colab *(Mặc định `False` để bạn kịp đọc mẫu; đổi `True` nếu chạy không canh máy)*

In [ ]:
import time

AUTO_DISCONNECT = False

if AUTO_DISCONNECT:
    print("✅ Hoàn tất! Chờ 15 giây đồng bộ Drive rồi tự ngắt runtime...")
    time.sleep(15)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Không thể gọi runtime.unassign(): {e}")
else:
    print("ℹ️ AUTO_DISCONNECT=False — Đọc xong sample nhớ tắt runtime thủ công để không tốn Compute Units.")
